# Module A: Retail Revenue Intelligence
## RISK Enterprise Decision Intelligence Platform
**Role**: Principal Data Scientist & Senior Data Analyst  
**Dataset**: Superstore Retail Sales (data/raw/superstore_sales.csv)  
**Objective**: Comprehensive empirical investigation of sales volume, profitability distributions, discount sensitivity, customer segments, regional drivers, and seasonal velocity.


### 1. Ingestion & Multi-Table Disaggregation
We load the raw dataset through our resilient loader (src/loaders.py) which isolates the Orders table from the appended People and Returns blocks.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is accessible
sys.path.insert(0, '..')

from src.loaders import load_superstore_raw
from src.cleaning import clean_superstore_orders
from src.metrics import calculate_retail_kpis, aggregate_retail_by_dimension
from src.visualization import set_reporting_style, PALETTE

set_reporting_style()

# Load and clean
orders_raw, people_raw, returns_raw = load_superstore_raw('../data/raw/superstore_sales.csv')
df = clean_superstore_orders(orders_raw, returns_raw)
print(f'Processed Orders: {df.shape[0]:,} records across {df.shape[1]} features.')


### 2. Verified Executive KPI Scorecard
Summary of audited commercial metrics across the entire enterprise retail portfolio.


In [ ]:
kpis = calculate_retail_kpis(df)
for k, v in kpis.items():
    if isinstance(v, float):
        print(f'{k:<26}: {v:,.2f}')
    else:
        print(f'{k:<26}: {v:,}')


### 3. Category & Sub-Category Profitability Diagnostics
Analyzing the revenue and margin divergence across product hierarchies.


In [ ]:
cat_summary = aggregate_retail_by_dimension(df, 'Category')
display(cat_summary[['Category', 'Total_Sales', 'Total_Profit', 'Profit_Margin_Pct', 'Sales_Share_Pct']])

subcat_summary = aggregate_retail_by_dimension(df, 'Sub-Category')
display(subcat_summary[['Sub-Category', 'Total_Sales', 'Total_Profit', 'Profit_Margin_Pct']].sort_values('Total_Profit'))


### 4. Discount Sensitivity & Margin Destruction
Evaluating the non-linear relationship between commercial discounting and profitability.


In [ ]:
discount_summary = df.groupby('Discount Band', observed=False).agg(
    Orders=('Order ID', 'nunique'),
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Margin_Pct=('Profit', lambda p: (p.sum() / df.loc[p.index, 'Sales'].sum()) * 100)
).reset_index()
display(discount_summary)


### 5. Regional & Customer Segment Matrix
Mapping the geographic and behavioral concentration of commercial profits.


In [ ]:
region_summary = aggregate_retail_by_dimension(df, 'Region')
display(region_summary[['Region', 'Total_Sales', 'Total_Profit', 'Profit_Margin_Pct']])

segment_summary = aggregate_retail_by_dimension(df, 'Segment')
display(segment_summary[['Segment', 'Total_Sales', 'Total_Profit', 'Profit_Margin_Pct']])


### 6. Seasonality and Quarterly Growth
Evaluating monthly velocity, Q4 holiday surges, and YoY trajectory.


In [ ]:
annual_trend = df.groupby('Year').agg(
    Annual_Sales=('Sales', 'sum'),
    Annual_Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique')
).reset_index()
annual_trend['Profit_Margin_Pct'] = (annual_trend['Annual_Profit'] / annual_trend['Annual_Sales']) * 100
annual_trend['YoY_Sales_Growth_Pct'] = annual_trend['Annual_Sales'].pct_change() * 100
display(annual_trend)
